 # Spark + Iceberg 

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
import time

spark = (
    SparkSession.builder
    .appName("Lesson24-Spark-Iceberg")
    .master("spark://spark-master:7077") #spark://spark-master:7077
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://matrix/warehouse")
    .getOrCreate()
)
spark.version

'3.5.2'

In [2]:
spark.sql("SHOW DATABASES IN local").show()


+---------+
|namespace|
+---------+
|       db|
+---------+



## 1. Iceberg Tables 


In [2]:
# spark.sql("CREATE DATABASE IF NOT EXISTS local.db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.db.orders (
    order_id STRING,
    customer_id STRING,
    amount DOUBLE,
    order_date DATE,
    region STRING
)
USING iceberg
PARTITIONED BY (days(order_date))
""")

spark.sql("SELECT * FROM local.db.orders").show()


+--------+-----------+------+----------+------+
|order_id|customer_id|amount|order_date|region|
+--------+-----------+------+----------+------+
+--------+-----------+------+----------+------+



## 2. Write Operations 


In [3]:
import random
from datetime import date, timedelta

random.seed(24)
regions = ["Baki", "Ganja", "Sumqayit"]
rows = []
for i in range(1, 101):
    rows.append((
        f"O{i:04d}", f"CU{(i % 20)+1:03d}",
        round(random.uniform(20, 500), 2),
        date(2026, 1, 1) + timedelta(days=random.randint(0, 59)),
        random.choice(regions)
    ))

initial_df = spark.createDataFrame(rows, ["order_id", "customer_id", "amount", "order_date", "region"])

initial_df.writeTo("local.db.orders").append()
print("append sonrası sətir sayı:", spark.table("local.db.orders").count())

append sonrası sətir sayı: 100


In [4]:
jan5_correction = initial_df.filter(F.col("order_date") == date(2026, 1, 5)) \
    .withColumn("amount", F.col("amount") + 1000) 

jan5_correction.writeTo("local.db.orders").overwritePartitions()
spark.table("local.db.orders").filter(F.col("order_date") == date(2026, 1, 5)).show(5)


+--------+-----------+------+----------+------+
|order_id|customer_id|amount|order_date|region|
+--------+-----------+------+----------+------+
|   O0086|      CU007|1208.5|2026-01-05|  Baki|
+--------+-----------+------+----------+------+



## 3. MERGE INTO 



In [5]:
staging_rows = []
for i in range(1, 11):
    staging_rows.append((f"O{i:04d}", f"CU{(i % 20)+1:03d}", 999.99, date(2026, 1, 1), "Baki"))
for i in range(101, 106):
    staging_rows.append((f"O{i:04d}", f"CU{(i % 20)+1:03d}", round(random.uniform(20, 500), 2), date(2026, 3, 1), "Ganja"))

staging_df = spark.createDataFrame(staging_rows, ["order_id", "customer_id", "amount", "order_date", "region"])
staging_df.createOrReplaceTempView("staging_orders")
staging_df.show()

+--------+-----------+------+----------+------+
|order_id|customer_id|amount|order_date|region|
+--------+-----------+------+----------+------+
|   O0001|      CU002|999.99|2026-01-01|  Baki|
|   O0002|      CU003|999.99|2026-01-01|  Baki|
|   O0003|      CU004|999.99|2026-01-01|  Baki|
|   O0004|      CU005|999.99|2026-01-01|  Baki|
|   O0005|      CU006|999.99|2026-01-01|  Baki|
|   O0006|      CU007|999.99|2026-01-01|  Baki|
|   O0007|      CU008|999.99|2026-01-01|  Baki|
|   O0008|      CU009|999.99|2026-01-01|  Baki|
|   O0009|      CU010|999.99|2026-01-01|  Baki|
|   O0010|      CU011|999.99|2026-01-01|  Baki|
|   O0101|      CU002| 156.7|2026-03-01| Ganja|
|   O0102|      CU003|298.07|2026-03-01| Ganja|
|   O0103|      CU004|202.92|2026-03-01| Ganja|
|   O0104|      CU005| 27.74|2026-03-01| Ganja|
|   O0105|      CU006|425.71|2026-03-01| Ganja|
+--------+-----------+------+----------+------+



In [6]:

spark.sql("""
MERGE INTO local.db.orders t
USING staging_orders s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET t.amount = s.amount, t.order_date = s.order_date, t.region = s.region
WHEN NOT MATCHED THEN INSERT (order_id, customer_id, amount, order_date, region)
    VALUES (s.order_id, s.customer_id, s.amount, s.order_date, s.region)
""")

print("MERGE sonrası cəmi sətir sayı (100 + 5 yeni = 105 olmalıdır):", spark.table("local.db.orders").count())
spark.table("local.db.orders").filter(F.col("order_id").isin("O0001", "O0105")).show()

MERGE sonrası cəmi sətir sayı (100 + 5 yeni = 105 olmalıdır): 105
+--------+-----------+------+----------+------+
|order_id|customer_id|amount|order_date|region|
+--------+-----------+------+----------+------+
|   O0001|      CU002|999.99|2026-01-01|  Baki|
|   O0105|      CU006|425.71|2026-03-01| Ganja|
+--------+-----------+------+----------+------+



Schema Evolution 


In [7]:
spark.sql("ALTER TABLE local.db.orders ADD COLUMN discount_pct DOUBLE")
spark.sql("ALTER TABLE local.db.orders RENAME COLUMN customer_id TO client_id")
spark.table("local.db.orders").printSchema()

root
 |-- order_id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- discount_pct: double (nullable = true)



In [ ]:
spark.sql("ALTER TABLE local.db.orders SET TBLPROPERTIES ('write.spark.accept-any-schema'='true')")

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

extra_col_schema = StructType([
    StructField("order_id", StringType()),
    StructField("client_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("order_date", DateType()),
    StructField("region", StringType()),
    StructField("discount_pct", DoubleType()),   
    StructField("channel", StringType()),
])

extra_col_df = spark.createDataFrame(
    [("O0999", "CU001", 150.0, date(2026, 4, 1), "Baki", None, "mobile")],
    schema=extra_col_schema
)

extra_col_df.writeTo("local.db.orders").option("mergeSchema", "true").append()
spark.table("local.db.orders").printSchema()
spark.table("local.db.orders").filter(F.col("order_id") == "O0999").show()

## 5. Partition Evolution


In [ ]:
spark.sql("SELECT * FROM local.db.orders.partitions").show(5, truncate=False)

spark.sql("ALTER TABLE local.db.orders REPLACE PARTITION FIELD days(order_date) WITH months(order_date)")

new_batch_schema = StructType([
    StructField("order_id", StringType()),
    StructField("client_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("order_date", DateType()),
    StructField("region", StringType()),
    StructField("discount_pct", DoubleType()),
    StructField("channel", StringType()),
])

new_batch = spark.createDataFrame(
    [("O2000", "CU005", 88.0, date(2026, 5, 15), "Sumqayit", None, "web")],
    schema=new_batch_schema
)
new_batch.writeTo("local.db.orders").append()

spark.sql("SELECT * FROM local.db.orders.partitions").show(10, truncate=False)

In [ ]:
spark.sql("SELECT * FROM local.db.orders.partitions").printSchema()

## 6. Time Travel 


In [8]:
snapshots = spark.sql("SELECT snapshot_id, committed_at, operation FROM local.db.orders.snapshots ORDER BY committed_at").collect()
for row in snapshots:
    print(row.snapshot_id, row.committed_at, row.operation)

first_snapshot_id = snapshots[0].snapshot_id
spark.sql(f"SELECT COUNT(*) as cnt FROM local.db.orders VERSION AS OF {first_snapshot_id}").show()

spark.table("local.db.orders").count()

6450490520137400138 2026-09-22 16:39:31.897000 append
375184038097158952 2026-09-22 16:40:35.710000 overwrite
5314143270619270790 2026-09-22 16:41:07.519000 overwrite
+---+
|cnt|
+---+
|100|
+---+



105

## 7. Metadata Queries 


In [9]:
print("=== Manifest files ===")
spark.sql("SELECT path, partition_spec_id, added_data_files_count FROM local.db.orders.manifests").show(truncate=False)

print("=== Data files ===")
spark.sql("SELECT file_path, file_format, record_count, file_size_in_bytes FROM local.db.orders.files").show(5, truncate=False)

print("=== table history ===")
spark.sql("SELECT * FROM local.db.orders.history").show(truncate=False)

=== Manifest files ===
+--------------------------------------------------------------------------------------+-----------------+----------------------+
|path                                                                                  |partition_spec_id|added_data_files_count|
+--------------------------------------------------------------------------------------+-----------------+----------------------+
|s3a://matrix/warehouse/db/orders/metadata/d99e774b-e8b6-491b-8d30-687078264717-m1.avro|0                |8                     |
|s3a://matrix/warehouse/db/orders/metadata/8725b5b6-e116-4f5f-b006-336dfe78e40e-m1.avro|0                |1                     |
|s3a://matrix/warehouse/db/orders/metadata/d99e774b-e8b6-491b-8d30-687078264717-m0.avro|0                |0                     |
+--------------------------------------------------------------------------------------+-----------------+----------------------+

=== Data files ===
+----------------------------------------------

## 8. Incremental Reads 


In [ ]:
snap_ids = [r.snapshot_id for r in spark.sql("SELECT snapshot_id, operation FROM local.db.orders.snapshots ORDER BY committed_at").collect()]
append_snapshots = [r for r in spark.sql("SELECT snapshot_id, committed_at, operation FROM local.db.orders.snapshots ORDER BY committed_at").collect()]

for r in append_snapshots:
    print(r.snapshot_id, r.operation)

In [ ]:
# İki APPEND snapshot 
try:
    append_ids = [r.snapshot_id for r in append_snapshots if r.operation == "append"]
    if len(append_ids) >= 2:
        inc_df = (
            spark.read.format("iceberg")
            .option("start-snapshot-id", append_ids[0])
            .option("end-snapshot-id", append_ids[-1])
            .load("local.db.orders")
        )
        print("Incremental :", inc_df.count())
    else:
        print("not append file")
except Exception as e:
    print("merge ", type(e).__name__)

In [21]:
print("Snapshot sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders.snapshots").collect()[0].cnt)
print("Data fayl sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders.files").collect()[0].cnt)

Snapshot sayı: 1
Data fayl sayı: 47


In [10]:
print("Snapshot sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders.snapshots").collect()[0].cnt)
print("Data fayl sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders.files").collect()[0].cnt)

Snapshot sayı: 3
Data fayl sayı: 47


In [20]:
import datetime

now = datetime.datetime.now().isoformat(sep=" ", timespec="seconds")

spark.sql(f"""
CALL local.system.expire_snapshots(
    table => 'db.orders',
    older_than => TIMESTAMP '{now}',
    retain_last => 1
)
""").show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|0                       |0                                  |0                                  |3                           |1                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [19]:
# 2) remove_orphan_files
spark.sql("""
CALL local.system.remove_orphan_files(
    table => 'db.orders'
)
""").show(truncate=False)

+--------------------+
|orphan_file_location|
+--------------------+
+--------------------+



In [18]:
# 3) rewrite_data_files 
spark.sql("""
CALL local.system.rewrite_data_files(
    table => 'db.orders'
)
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|0                         |0                     |0                    |0                      |0                         |
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [17]:
# 4) rewrite_manifests 
spark.sql("""
CALL local.system.rewrite_manifests(
    table => 'db.orders'
)
""").show(truncate=False)

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|0                        |0                    |
+-------------------------+---------------------+

